

# Install dependancies

In [ ]:
%%shell
apt-get update
apt-get install -y build-essential python3-dev automake cmake git flex bison libglib2.0-dev libpixman-1-dev python3-setuptools cargo libgtk-3-dev
# try to install llvm-18 and install the distro default if that fails
apt-get install -y lld-18 llvm-18 llvm-18-dev clang-18 || sudo apt-get install -y lld llvm llvm-dev clang
apt-get install -y gcc-$(gcc --version|head -n1|sed 's/\..*//'|sed 's/.* //')-plugin-dev libstdc++-$(gcc --version|head -n1|sed 's/\..*//'|sed 's/.* //')-dev
apt-get install -y meson ninja-build # for QEMU mode
apt-get install -y cpio libcapstone-dev # for Nyx mode
apt-get install -y wget curl # for Frida mode
apt-get install -y python3-pip # for Unicorn mode

#update rustc
apt remove rustc
curl https://sh.rustup.rs -sSf | sh -s -- -y
. "$HOME/.cargo/env"
rustc --version


# Build AFLplusplus

### Install dependancies for python bindings and create a virtual environment

In [ ]:
%%shell
cd /content
pip install virtualenv
virtualenv fuzzingenv
source fuzzingenv/bin/activate
pip install setuptools
pip install pyelftools


### update rust and install afl++

In [ ]:

%%shell
cd /content
. "$HOME/.cargo/env"
source fuzzingenv/bin/activate
git clone https://github.com/AFLplusplus/AFLplusplus
cd AFLplusplus
git submodule update --init
make clean
make distrib NO_NYX=1 NO_CORESIGHT=1 NO_QEMU=1 NO_FRIDA=1
make install
# disable the /proc/sys/kernel/core_pattern check
# export AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1

# Test afl-fuzz binary is working correclty

In [ ]:
%env AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1
!afl-fuzz

 # Build a binary with AFLplusplus instrumentation

### Mount the drive as a local filesystem

In [3]:
!git clone https://github.com/st-onlinetraining/fuzzing.git
import os
os.chdir("/content/fuzzing")
!ls

Cloning into 'fuzzing'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 48 (delta 9), reused 28 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 386.69 KiB | 7.73 MiB/s, done.
Resolving deltas: 100% (9/9), done.
 dict			  input_basic	 out	     rustup.sh
'Fuzzing handson.ipynb'   input_corpus	 output      vulnerable-arm
 harness.py		  Makefile	 README.md   vulnerable.c


### Build the binary with afl-gcc to inject instrumentation

In [4]:
!make clean
!make

rm -f vulnerable
rm -f vulnerable-arm
afl-gcc-fast -g -w  vulnerable.c -o vulnerable 	
make: afl-gcc-fast: No such file or directory
make: *** [Makefile:14: vulnerable-afl] Error 127


### Test the binary with a random input

In [ ]:
!echo "Hello" | ./vulnerable

### Check the binary crash with the magic inputs

In [ ]:
!echo "foo!" | ./vulnerable

In [ ]:
!echo "GOODFUZZER" | ./vulnerable

# Start fuzzing with basic inputs

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -i ./input_basic -o ./output ./vulnerable

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

# Use better input corpus

In this cas the input corpus already passes 3 out of the 4 checks required to reach the crash.

The fuzzer only needs to find one addicitonal byte value.

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -i ./input_corpus -o ./output ./vulnerable

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

# Use a dictionnary

The dictionary dict.dct contains various keywords. Keywords are inserted by the fuzzer during the mutations.

A combination of 2 keywords is required in the input to reach the crash.

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -x dict/dict.dct -i ./input_corpus -o ./output ./vulnerable

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

[link text](https://)Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

Replay our crash test cases

In [ ]:
!(cat ./output/default/crashes/id:000001,sig:06,src:000003,time:38067,execs:13913,op:havoc,rep:55 ; echo "") | ./vulnerable

# Fuzz an arm binary

### Install the cross compiler

In [ ]:
!apt install gcc-arm-linux-gnueabihf

### Build the arm target binary

In [ ]:
!make clean
!make vulnerable-arm

### The symbols can be retrieved from the binary

In [ ]:
!nm --print-size ./vulnerable-arm | grep -E " main$"
!arm-linux-gnueabihf-objdump -d ./vulnerable-arm
!arm-linux-gnueabihf-objdump --disassemble=main ./vulnerable-arm
!arm-linux-gnueabihf-objdump --disassemble=main ./vulnerable-arm | grep -E "pop.*pc" | cut -f 1 | sed s/://

### Run the fuzzing using the unicorn emulator
The file harness.py contains the code that emulates the execution environment for the target binary
- Maps the memory
- Copies the code in memory
- Hook (stub) the libc functions
- Copies the afl data in the input buffer at each execution

In [ ]:
%%shell
rm -r ./output
source /content/fuzzingenv/bin/activate
afl-fuzz -U -i ./input_corpus -o ./output -- python3 harness.py @@ ./vulnerable-arm
#AFL_NO_UI=1 AFL_DEBUG=1 AFL_DEBUG_CHILD=1 afl-fuzz -U -i ./input_corpus -o ./output -- python3 harness.py @@ ./vulnerable-arm